# Perturbation & Basis Methods — Validation

Internal correctness checks, run phase by phase. See
`Perturbation_and_Basis_Methods_Plan.md` for the full design rationale.

## Phase 1 — Basis sets & hydrogen wavefunctions

Checks that `basis.py`'s two eigenbases (infinite square well, harmonic
oscillator) are genuinely orthonormal and have the right energies, and that
`hydrogen.py`'s radial and angular wavefunctions are correctly normalized,
mutually orthogonal, and have the right energies -- all against known analytic
results, independent of anything in `TDSE_Solver` (though the box/HO energy
formulas were already confirmed there too, in `stationary_states.py`'s Phase 4
validation, via a completely different numerical method).

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
from scipy.integrate import simpson
import basis as bs
import hydrogen as hyd

results = []
def check(name, cond, detail=""):
    results.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [2]:
# --- Check 1a: box_basis is orthonormal, energies match analytic ---
L, N, num_modes = 5.0, 4000, 8
x_box = np.linspace(0, L, N)
box = bs.box_basis(x_box, L, num_modes)
gram = simpson(box.eigenfunctions[:, None, :] * box.eigenfunctions[None, :, :], x=x_box, axis=2)
err_gram = np.max(np.abs(gram - np.eye(num_modes)))
check("1a box_basis orthonormal", err_gram < 1e-8, f"max|Gram-I|={err_gram:.2e}")

analytic_E = (np.arange(1, num_modes + 1) * np.pi / L) ** 2 / 2
err_E = np.max(np.abs(box.energies - analytic_E) / analytic_E)
check("1a box_basis energies match (n*pi/L)^2/2", err_E < 1e-12, f"max rel err={err_E:.2e}")


PASS: 1a box_basis orthonormal  max|Gram-I|=2.60e-13
PASS: 1a box_basis energies match (n*pi/L)^2/2  max rel err=0.00e+00


In [3]:
# --- Check 1b: harmonic_basis is orthonormal, energies match analytic ---
L2, N2, omega, num_modes2 = 30.0, 4000, 1.0, 8
x_ho = np.linspace(-L2 / 2, L2 / 2, N2)
ho = bs.harmonic_basis(x_ho, omega, num_modes2)
gram2 = simpson(ho.eigenfunctions[:, None, :] * ho.eigenfunctions[None, :, :], x=x_ho, axis=2)
err_gram2 = np.max(np.abs(gram2 - np.eye(num_modes2)))
check("1b harmonic_basis orthonormal", err_gram2 < 1e-8, f"max|Gram-I|={err_gram2:.2e}")

analytic_E2 = omega * (np.arange(num_modes2) + 0.5)
err_E2 = np.max(np.abs(ho.energies - analytic_E2) / analytic_E2)
check("1b harmonic_basis energies match omega*(n+1/2)", err_E2 < 1e-12, f"max rel err={err_E2:.2e}")


PASS: 1b harmonic_basis orthonormal  max|Gram-I|=1.11e-15
PASS: 1b harmonic_basis energies match omega*(n+1/2)  max rel err=0.00e+00


In [4]:
# --- Check 1c: hydrogen radial wavefunctions normalized & orthogonal
# across n (same l), energies match -1/(2n^2) ---
r = np.linspace(1e-6, 60.0, 20000)
max_norm_err, max_ortho_err = 0.0, 0.0
for l in (0, 1):
    ns = [n for n in (1, 2, 3) if n > l]
    Rs = {n: hyd.radial_wavefunction(n, l, r) for n in ns}
    for n in ns:
        norm_val = simpson(Rs[n] ** 2 * r ** 2, x=r)
        max_norm_err = max(max_norm_err, abs(norm_val - 1.0))
    for i in range(len(ns)):
        for j in range(i + 1, len(ns)):
            ov = simpson(Rs[ns[i]] * Rs[ns[j]] * r ** 2, x=r)
            max_ortho_err = max(max_ortho_err, abs(ov))
check("1c hydrogen radial functions normalized", max_norm_err < 1e-4, f"max|norm-1|={max_norm_err:.2e}")
check("1c hydrogen radial functions orthogonal across n", max_ortho_err < 1e-4, f"max|<Rn1|Rn2>|={max_ortho_err:.2e}")

E_numeric = np.array([hyd.energy(n) for n in (1, 2, 3)])
E_analytic = np.array([-1 / (2 * n ** 2) for n in (1, 2, 3)])
check("1c hydrogen energies match -1/(2n^2)", np.allclose(E_numeric, E_analytic), f"{E_numeric}")


PASS: 1c hydrogen radial functions normalized  max|norm-1|=2.07e-10
PASS: 1c hydrogen radial functions orthogonal across n  max|<Rn1|Rn2>|=7.64e-12
PASS: 1c hydrogen energies match -1/(2n^2)  [-0.5        -0.125      -0.05555556]


In [5]:
# --- Check 1d: hydrogen angular wavefunctions orthonormal (Gauss-Legendre
# quadrature in cos(theta) -- exact for these smooth functions, unlike a
# naive uniform grid which converges only as O(1/N) near the poles) ---
deg = 30
u, w_u = np.polynomial.legendre.leggauss(deg)
theta_nodes = np.arccos(u)
n_phi = 64
phi_nodes = np.linspace(0, 2 * np.pi, n_phi, endpoint=False)
dphi = 2 * np.pi / n_phi

pairs = [(0, 0), (1, -1), (1, 0), (1, 1), (2, -2), (2, -1), (2, 0), (2, 1), (2, 2)]
TH, PH = np.meshgrid(theta_nodes, phi_nodes, indexing='ij')
Yv = {p: hyd.angular_wavefunction(p[0], p[1], TH, PH) for p in pairs}

gram3 = np.zeros((len(pairs), len(pairs)), dtype=complex)
for i, a in enumerate(pairs):
    for j, b in enumerate(pairs):
        gram3[i, j] = np.sum(w_u[:, None] * np.conj(Yv[a]) * Yv[b]) * dphi
err_gram3 = np.max(np.abs(gram3 - np.eye(len(pairs))))
check("1d hydrogen angular functions orthonormal", err_gram3 < 1e-10, f"max|Gram-I|={err_gram3:.2e}")


PASS: 1d hydrogen angular functions orthonormal  max|Gram-I|=9.55e-15


In [6]:
# --- Check 1e: real p-orbitals (real_p_orbitals) are actually real ---
n_p = 2
r_p = np.linspace(1e-6, 30, 60)
theta_p = np.linspace(1e-6, np.pi - 1e-6, 60)
phi_p = np.linspace(0, 2 * np.pi, 60, endpoint=False)
R3, TH3, PH3 = np.meshgrid(r_p, theta_p, phi_p, indexing='ij')
px, py, pz = hyd.real_p_orbitals(n_p, R3, TH3, PH3)
max_imag = max(np.max(np.abs(px.imag)), np.max(np.abs(py.imag)), np.max(np.abs(pz.imag)))
check("1e real p-orbitals have zero imaginary part", max_imag < 1e-12, f"max|Im|={max_imag:.2e}")


PASS: 1e real p-orbitals have zero imaginary part  max|Im|=0.00e+00


In [7]:
n_pass1 = sum(1 for _, ok, _ in results if ok)
print(f"\n{n_pass1}/{len(results)} Phase 1 checks passed")
assert n_pass1 == len(results), "Phase 1 validation failed"



9/9 Phase 1 checks passed


## Phase 2 — Basis expansion & time evolution

Two checks: (1) truncation error (`1-sum(|c_n|^2)`, the norm missing from a
finite mode sum) should shrink -- fast, since these are smooth target
functions -- as more modes are kept, for both bases; (2) a genuine physics
check with no free parameters: a Gaussian wavepacket whose width is *exactly*
matched to the harmonic oscillator's natural length (`sigma=1/sqrt(omega)`) is
a textbook coherent state, so `<x>(t)` must follow the *exact classical
trajectory* `x0*cos(omega*t)` -- not approximately, exactly (in the limit of
enough retained modes) -- a much stronger test than just "looks like it
oscillates".

In [8]:
results2 = []
def check2(name, cond, detail=""):
    results2.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [9]:
# --- Check 2a: box-basis truncation error shrinks with more modes ---
L3, N3 = 10.0, 2000
x3 = np.linspace(0, L3, N3)
xc, sigma3 = 5.0, 0.5
psi_target = np.exp(-(x3 - xc) ** 2 / (2 * sigma3 ** 2)) / (np.pi * sigma3 ** 2) ** 0.25

mode_counts = (5, 10, 20, 40, 80)
missing_norms = []
for nm in mode_counts:
    box3 = bs.box_basis(x3, L3, nm)
    c = bs.project(box3, psi_target)
    missing_norms.append(1 - np.sum(c ** 2))
    print(f"  num_modes={nm}: missing norm = {missing_norms[-1]:.3e}")

check2("2a truncation error shrinks (or hits the machine-precision floor) as modes increase",
       all(missing_norms[i] >= missing_norms[i + 1] for i in range(len(missing_norms) - 1)),
       f"{[f'{m:.2e}' for m in missing_norms]}")
check2("2a truncation error is tiny with enough modes", missing_norms[-1] < 1e-10, f"{missing_norms[-1]:.2e}")

box_final = bs.box_basis(x3, L3, mode_counts[-1])
c_final = bs.project(box_final, psi_target)
recon = bs.reconstruct(box_final, c_final)
max_recon_err = np.max(np.abs(recon - psi_target))
check2("2a reconstruction matches target to truncation-limited tolerance", max_recon_err < 1e-8, f"max err={max_recon_err:.2e}")


  num_modes=5: missing norm = 1.790e-01
  num_modes=10: missing norm = 2.510e-02
  num_modes=20: missing norm = 7.504e-06
  num_modes=40: missing norm = 1.110e-16
  num_modes=80: missing norm = 1.110e-16
PASS: 2a truncation error shrinks (or hits the machine-precision floor) as modes increase  ['1.79e-01', '2.51e-02', '7.50e-06', '1.11e-16', '1.11e-16']
PASS: 2a truncation error is tiny with enough modes  1.11e-16
PASS: 2a reconstruction matches target to truncation-limited tolerance  max err=1.27e-15


In [10]:
# --- Check 2b: harmonic-oscillator coherent state matches the exact
# classical trajectory x0*cos(omega*t), with no fitting ---
omega4 = 1.0
sigma4 = 1.0 / np.sqrt(omega4)  # exact coherent-state width
x0_4 = 3.0
L4, N4 = 30.0, 2000
x4 = np.linspace(-L4 / 2, L4 / 2, N4)
psi0_4 = (2 * np.pi * sigma4 ** 2) ** (-0.25) * np.exp(-(x4 - x0_4) ** 2 / (4 * sigma4 ** 2))

num_modes4 = 40
ho4 = bs.harmonic_basis(x4, omega4, num_modes4)
c0_4 = bs.project(ho4, psi0_4)
missing4 = 1 - np.sum(np.abs(c0_4) ** 2)
print(f"coherent-state truncation: missing norm = {missing4:.3e} (num_modes={num_modes4})")

T4 = 2 * 2 * np.pi / omega4  # 2 classical periods
t4 = np.linspace(0, T4, 300)
ct4 = bs.evolve(ho4, c0_4, t4)
psi_t4 = ct4 @ ho4.eigenfunctions
density_t4 = np.abs(psi_t4) ** 2
x_mean_t4 = simpson(x4[None, :] * density_t4, x=x4, axis=1) / simpson(density_t4, x=x4, axis=1)
x_predicted_t4 = x0_4 * np.cos(omega4 * t4)

max_err4 = np.max(np.abs(x_mean_t4 - x_predicted_t4))
check2("2b coherent-state truncation error is tiny", missing4 < 1e-6, f"{missing4:.2e}")
check2("2b <x>(t) matches exact classical trajectory x0*cos(wt)", max_err4 < 1e-3, f"max err={max_err4:.2e} (amplitude={x0_4})")


coherent-state truncation: missing norm = 4.782e-10 (num_modes=40)
PASS: 2b coherent-state truncation error is tiny  4.78e-10
PASS: 2b <x>(t) matches exact classical trajectory x0*cos(wt)  max err=4.80e-09 (amplitude=3.0)


In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
Path('media').mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t4, x_mean_t4, label='simulated <x>(t) (basis expansion)', color='C0')
ax.plot(t4, x_predicted_t4, '--', label='exact classical trajectory x0*cos(wt)', color='C3')
ax.set_xlabel('t')
ax.set_ylabel('<x>')
ax.set_title('Harmonic oscillator coherent state: exact classical motion')
ax.legend()
fig.tight_layout()
fig.savefig('media/coherent_state_trajectory.png', dpi=150)
plt.show()


C:\Users\Hasan's Laptop\AppData\Local\Temp\ipykernel_9292\1937005305.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
n_pass2 = sum(1 for _, ok, _ in results2 if ok)
print(f"\n{n_pass2}/{len(results2)} Phase 2 checks passed")
assert n_pass2 == len(results2), "Phase 2 validation failed"



5/5 Phase 2 checks passed
